# Laboratorio guiado: Delta Lake sobre HDFS e SeaweedFS

Este laboratorio desenvolve o percorrido do [tema 08](../08.Delta-Lake.md): crear unha táboa, consultala, actualizar datos con MERGE e recuperar unha versión anterior. Inclúe exemplos separados de evolución do esquema e almacenamento S3.

## 1. Contorno e sesión

O clúster base debe estar iniciado e JupyterLab accesible, como explica o [contorno de traballo](../02.construcion-escenarios.md#contorno-de-traballo-do-curso). Para a parte S3 tamén debe estar iniciado SeaweedFS e existir o bucket `spark`, segundo o [tema 07](../07.s3-seaweed.md).

Os bloques Python execútanse nun notebook, en orde; os bloques Bash, no terminal de JupyterLab; os comandos Docker, en PowerShell no cartafol `big-data-cluster`. Os datos créanse no propio notebook: non se necesita un ficheiro externo.

A sesión usa a configuración existente:

Executa as celas Python en orde. Os comandos Bash e PowerShell dos bloques explicativos execútanse nos terminais indicados, fóra das celas Python.


In [ ]:
# 1. Contorno e sesión
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("delta-lake").getOrCreate()


## 2. Delta Lake sobre HDFS

Os exemplos traballan nun cartafol de HDFS chamado **`delta`**, situado en `/tmp/delta`. Antes de crear as táboas, ese cartafol pode prepararse desde o **terminal de JupyterLab** con:

```bash
hdfs dfs -mkdir -p /tmp/delta
```

`-p` permite crear a ruta aínda que algún dos seus directorios xa exista. As táboas dos exemplos quedarán baixo ese cartafol, que debe ser escribible pola aplicación.

O NameNode xa está definido na configuración do clúster que usa Spark. Por iso as rutas poden escribirse como **`/tmp/delta/...`**, sen repetir `hdfs://namenode:9000`.

### 2.1. Datos e ruta de exemplo

O conxunto inicial ten catro rexistros de temperatura. Para simplificar, a columna `data` contén datas como texto.

In [ ]:
# 2.1. Datos e ruta de exemplo
from uuid import uuid4

datos = [
    ("2024-01-01", "A Coruña", 11.2),
    ("2024-01-01", "Vigo", 13.5),
    ("2024-01-02", "A Coruña", 9.8),
    ("2024-01-02", "Vigo", 14.1),
]

esquema = "data STRING, cidade STRING, temperatura DOUBLE"
df = spark.createDataFrame(datos, esquema)
df.show()

ruta_delta = f"/tmp/delta/meteo/{uuid4().hex}"
print("Táboa HDFS:", ruta_delta)


Cada execución crea unha ruta diferente dentro do cartafol `delta`. Así, a versión 0 corresponde á creación desta táboa e os exemplos non sobrescriben resultados anteriores.

### 2.2. Crear a táboa

In [ ]:
# 2.2. Crear a táboa
df.write.format("delta").mode("errorifexists").save(ruta_delta)


Esta escritura crea os Parquet e `_delta_log`. `errorifexists` evita substituír accidentalmente unha táboa existente.

A estrutura pode consultarse desde o **terminal de JupyterLab**, onde están dispoñibles os comandos de Hadoop. Substituíndo `IDENTIFICADOR` polo valor da ruta impresa:

```bash
hdfs dfs -ls /tmp/delta/meteo/IDENTIFICADOR
hdfs dfs -ls /tmp/delta/meteo/IDENTIFICADOR/_delta_log
```

### 2.3. Ler e consultar

In [ ]:
# 2.3. Ler e consultar
df_delta = spark.read.format("delta").load(ruta_delta)
df_delta.show()

df_delta.createOrReplaceTempView("meteo")

spark.sql("""
    SELECT cidade, AVG(temperatura) AS media
    FROM meteo
    GROUP BY cidade
    ORDER BY cidade
""").show()


A vista temporal `meteo` permite consultar o DataFrame mediante SQL nesta sesión. Non crea unha táboa permanente nun catálogo.

### 2.4. Actualizar e inserir con MERGE

Un **upsert** combina actualización e inserción. Neste exemplo, o par **data + cidade** identifica o rexistro:

In [ ]:
# 2.4. Actualizar e inserir con MERGE
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, ruta_delta)

novos_datos = [
    ("2024-01-02", "A Coruña", 10.2),
    ("2024-01-03", "Ourense", 6.4),
]

df_novos = spark.createDataFrame(novos_datos, esquema)

(
    delta_table.alias("destino")
    .merge(
        df_novos.alias("orixe"),
        "destino.data = orixe.data AND destino.cidade = orixe.cidade",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

spark.read.format("delta").load(ruta_delta).orderBy("data", "cidade").show()


O rexistro de A Coruña do 2 de xaneiro pasa de **9.8 a 10.2** e aparece o novo rexistro de Ourense. O resultado ten **cinco filas**:

| data | cidade | temperatura |
|---|---|---|
| 2024-01-01 | A Coruña | 11.2 |
| 2024-01-01 | Vigo | 13.5 |
| 2024-01-02 | A Coruña | 10.2 |
| 2024-01-02 | Vigo | 14.1 |
| 2024-01-03 | Ourense | 6.4 |

`MERGE` non elimina duplicados automaticamente. A condición debe identificar ben os rexistros e os datos de entrada non deben conter varias filas que actualicen o mesmo rexistro de destino. [Actualizacións e MERGE](https://docs.delta.io/delta-update/).

### 2.5. Histórico e time travel

O histórico mostra as operacións confirmadas e as súas versións:

In [ ]:
# 2.5. Histórico e time travel
delta_table.history().show(truncate=False)


Despois da creación e do primeiro `MERGE`, as versións son **0** e **1**. A versión inicial pode consultarse así:

In [ ]:
# 2.5. Histórico e time travel
df_v0 = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(ruta_delta)
)

df_v0.orderBy("data", "cidade").show()


Esta lectura recupera as catro filas iniciais, coa temperatura **9.8** para A Coruña do 2 de xaneiro. Non modifica a versión actual.

Tamén existe a opción `timestampAsOf` para consultar o estado nun momento determinado. Ambas formas necesitan que se conserven o rexistro e os ficheiros da versión solicitada.

Repetir unha escritura con `overwrite` sobre unha táboa Delta existente crea outra versión; non reinicia o histórico nin cambia o significado da versión 0.

### 2.6. Unha nova carga diaria

O mesmo patrón permite incorporar datos doutro día e corrixir un valor anterior:

In [ ]:
# 2.6. Unha nova carga diaria
datos_hoxe = [
    ("2024-01-02", "Vigo", 13.9),
    ("2024-01-03", "A Coruña", 7.3),
    ("2024-01-03", "Vigo", 8.1),
]

df_hoxe = spark.createDataFrame(datos_hoxe, esquema)

(
    delta_table.alias("destino")
    .merge(
        df_hoxe.alias("orixe"),
        "destino.data = orixe.data AND destino.cidade = orixe.cidade",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

spark.read.format("delta").load(ruta_delta).orderBy("data", "cidade").show()
delta_table.history().show(truncate=False)


Tras esta carga hai **sete filas** e unha nova versión. O histórico permite identificar que operación introduciu cada cambio.

## 3. Validación e evolución do esquema

Estes exemplos usan unha táboa separada para non alterar a demostración anterior.

### 3.1. Un tipo incompatible

In [ ]:
# 3.1. Un tipo incompatible
from pyspark.errors import AnalysisException

ruta_esquema = f"/tmp/delta/esquema/{uuid4().hex}"
df.write.format("delta").mode("errorifexists").save(ruta_esquema)

df_incompatible = spark.createDataFrame(
    [("2024-01-03", "Lugo", "frío")],
    "data STRING, cidade STRING, temperatura STRING",
)

try:
    df_incompatible.write.format("delta").mode("append").save(ruta_esquema)
except AnalysisException as erro:
    print("Escritura rexeitada: temperatura debe ser DOUBLE.")
    print(str(erro))


A táboa espera unha temperatura numérica. Esta escritura intenta introducir unha columna de texto incompatible e non se confirma. Un valor numérico incorrecto para o dominio, en cambio, non sería rexeitado só polo esquema.

### 3.2. Engadir unha columna

`mergeSchema` permite aceptar unha nova columna nesta escritura concreta:

In [ ]:
# 3.2. Engadir unha columna
df_ampliado = spark.createDataFrame(
    [("2024-01-03", "Lugo", 8.0, "estacion-local")],
    "data STRING, cidade STRING, temperatura DOUBLE, fonte STRING",
)

(
    df_ampliado.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(ruta_esquema)
)

spark.read.format("delta").load(ruta_esquema).show()


As filas anteriores teñen `NULL` na columna `fonte`. Esta opción habilita a evolución para a operación indicada, sen activala globalmente para todas as táboas. [Validación e evolución do esquema](https://docs.delta.io/delta-batch/).

## 4. Delta Lake sobre SeaweedFS

### 4.1. Almacenamento e conexión

Neste escenario, os datos gárdanse no bucket existente **`spark`**. O acceso S3A usa a configuración do clúster descrita en [07.s3-seaweed.md](../07.s3-seaweed.md), co endpoint interno **`http://seaweedfs:8333`** e as credenciais do servidor.

Non é necesario editar `core-site.xml` no NameNode para estes exemplos. A conexión que necesita Spark debe estar dispoñible no driver e nos executores.

### 4.2. Escritura e lectura

O DataFrame inicial `df` pode gardarse como outra táboa Delta en SeaweedFS:

In [ ]:
# 4.2. Escritura e lectura
ruta_s3 = f"s3a://spark/delta/meteo/{uuid4().hex}"

df.write.format("delta").mode("errorifexists").save(ruta_s3)

df_s3 = spark.read.format("delta").load(ruta_s3)
df_s3.show()

print("Táboa S3:", ruta_s3)


Os métodos son os mesmos que en HDFS; cambia a ruta de almacenamento. `DeltaTable.forPath(spark, ruta_s3)` permite aplicar `MERGE` e consultar o histórico desta táboa.

### 4.3. Consulta dos obxectos

Na Admin UI de SeaweedFS, en [http://localhost:9001](http://localhost:9001), a vista **Buckets → spark → delta → meteo** mostra os prefixos creados. Dentro de cada un aparecen os Parquet e `_delta_log`.

O listado tamén pode realizarse desde PowerShell, no cartafol dos ficheiros Compose:

```powershell
docker compose -f compose.base.yml -f compose.seaweedfs.yml run --rm aws-cli s3 ls s3://spark/delta/meteo/ --recursive
```


## 5. Comprobación final

O percorrido completo deixa sete filas na táboa HDFS, catro na súa versión inicial e catro na táboa S3. A táboa separada de evolución do esquema ten cinco filas e a nova columna `fonte`.

In [ ]:
# 5. Comprobación final
assert spark.read.format("delta").load(ruta_delta).count() == 7
assert df_v0.count() == 4
assert spark.read.format("delta").load(ruta_s3).count() == 4
assert spark.read.format("delta").load(ruta_esquema).count() == 5


As comprobacións finais presupoñen que se executaron tamén as partes de esquema e S3. Unha repetición completa empeza nun notebook novo ou desde a creación dunha nova ruta. Repetir só unha carga pode cambiar o histórico e os recontos.

Os resultados indicados son os esperados; a execución completa no clúster segue pendente. Este laboratorio usa un único escritor. As condicións de concorrencia e retención están no [tema 08](../08.Delta-Lake.md).

A aplicación dos conceptos avalíase no [boletín Delta e Iceberg](../tarefas/tarefa-04-delta-iceberg.ipynb).